<a href="https://colab.research.google.com/github/Dharmendra0305/PRODIGY_GA_01/blob/main/Text%20Generation%20with%20GPT-2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Environment Setup**

* Check GPU Availability



In [29]:
import torch
torch.cuda.is_available()

True

* Install Core Libraries

In [30]:
!pip install transformers datasets torch

* Verify Installation

In [31]:
import transformers
import datasets
import torch

print("Transformers Version:", transformers.__version__)
print("Datasets Version:", datasets.__version__)
print("PyTorch Version:", torch.__version__)

Transformers Version: 5.0.0
Datasets Version: 4.0.0
PyTorch Version: 2.10.0+cu128


# **Data Preparation**

* Collect Text

In [52]:
from google.colab import files

uploaded = files.upload()

Saving Text_Generation_with_GPT_2.ipynb to Text_Generation_with_GPT_2.ipynb


* Load Text

In [33]:
with open("gpt.txt", "r", encoding = "utf-8") as f:
  text = f.read()

print(text[:500])

For GPT training purposes, a model is trained on large and diverse datasets containing text from books, websites, articles, and other sources. The training process helps the model learn patterns in language, grammar, context, and reasoning so it can understand user inputs and generate meaningful responses. During training, the model adjusts its internal parameters to predict the most appropriate next word or sentence based on the context. This enables the GPT model to assist with tasks such as a


* Tokenization

In [34]:
from transformers import AutoTokenizer

# Load GPT-2 tokenizer
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Assign EOS token as pad token
tokenizer.pad_token = tokenizer.eos_token

# Tokenize text
tokens = tokenizer(text, return_tensors = "pt")

print(tokens["input_ids"][0][:50])

tensor([ 1890,   402, 11571,  3047,  4959,    11,   257,  2746,   318,  8776,
          319,  1588,   290, 10084, 40522,  7268,  2420,   422,  3835,    11,
         9293,    11,  6685,    11,   290,   584,  4237,    13,   383,  3047,
         1429,  5419,   262,  2746,  2193,  7572,   287,  3303,    11, 23491,
           11,  4732,    11,   290, 14607,   523,   340,   460,  1833,  2836])


* Chunking

In [35]:
block_size = 128

# Convert text into token IDs
token_ids = tokenizer.encode(text)

# Break into chunks
chunks = [token_ids[i:i + block_size] for i in range(0, len(token_ids), block_size)]

print("Number of chunks:", len(chunks))
print("First chunk:", chunks[0])

Number of chunks: 1
First chunk: [1890, 402, 11571, 3047, 4959, 11, 257, 2746, 318, 8776, 319, 1588, 290, 10084, 40522, 7268, 2420, 422, 3835, 11, 9293, 11, 6685, 11, 290, 584, 4237, 13, 383, 3047, 1429, 5419, 262, 2746, 2193, 7572, 287, 3303, 11, 23491, 11, 4732, 11, 290, 14607, 523, 340, 460, 1833, 2836, 17311, 290, 7716, 11570, 9109, 13, 5856, 3047, 11, 262, 2746, 46094, 663, 5387, 10007, 284, 4331, 262, 749, 5035, 1306, 1573, 393, 6827, 1912, 319, 262, 4732, 13, 770, 13536, 262, 402, 11571, 2746, 284, 3342, 351, 8861, 884, 355, 18877, 2683, 11, 3597, 2695, 11, 11170, 10838, 11, 290, 6493, 2972, 3453, 864, 5479, 13]


In [36]:
from datasets import Dataset

dataset = Dataset.from_dict({"input_ids": chunks})
print(dataset)

Dataset({
    features: ['input_ids'],
    num_rows: 1
})


# **Fine-Tuning (Training)**



* Load the Base Model

In [37]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("gpt2")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


* Prepare the Dataset

In [38]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(tokenizer = tokenizer,
                                                mlm = False)           # causal LM (GPT-2) does not use masked language modeling

* Configure the Trainer

In [39]:
import os
os.environ["TENSORBOARD_LOGGING_DIR"] = "./logs"

In [40]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir = "./results",        # where to save checkpoints

    num_train_epochs = 3,            # train for 3 cycles

    per_device_train_batch_size = 2, # batch size per GPU

    save_steps = 500,                # save every 500 steps

    save_total_limit = 2,            # keep only last 2 checkpoints

    logging_steps = 100              # log every 100 steps
)

trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = dataset,
    data_collator = data_collator
)

* Execute Training

In [41]:
trainer.train()

Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3, training_loss=3.2240492502848306, metrics={'train_runtime': 8.4777, 'train_samples_per_second': 0.354, 'train_steps_per_second': 0.354, 'total_flos': 163817856000.0, 'train_loss': 3.2240492502848306, 'epoch': 3.0})

* Save the Model

In [42]:
trainer.save_model("./fine_tuned_gpt2")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

# **Generation & Evaluation**

* Load Fine-Tuned Model

In [43]:
model = AutoModelForCausalLM.from_pretrained("./fine_tuned_gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token         # Keep the padding fix

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

* Prepare a Prompt

In [44]:
prompt = "A model is trained on large and diverse"

* Tokenize the Prompt

In [45]:
inputs = tokenizer(prompt, return_tensors = "pt", padding = True, return_attention_mask = True)

* Generate Text

In [46]:
outputs = model.generate(
    inputs["input_ids"],
    attention_mask = inputs["attention_mask"],

    pad_token_id = tokenizer.eos_token_id,  # explicitly set pad token

    max_length = 150,                       # total length of generated text

    num_return_sequences = 1,               # how many completions to generate

    temperature = 0.9,                      # creativity (lower = more focused, higher = more random)

    top_p = 0.85,                           # nucleus sampling (controls diversity)

    repetition_penalty = 1.2,               # discourages loops

    do_sample = True                        # enables sampling instead of greedy decoding
)

print(tokenizer.decode(outputs[0], skip_special_tokens = True))

A model is trained on large and diverse populations in different parts of the world. The individual's social behavior can be influenced by cultural context, socioeconomic status or other factors such as family structure, education levels etc…
 "The concept of a 'model' refers to an organization that uses data from one population (the dataset) for purposes like demographic analysis."–Michael Hesse

"This approach has broad applicability throughout all industries – retail management firms use it when developing customer-centric products where they are able quickly identify customers with similar characteristics; service providers deploy its tools at events including conferences about their product offerings but also market research surveys regarding brand identity issues".—Andrew Warshaw & Scott Oakeshott


In [53]:
!ls

fine_tuned_gpt2  results      Text_Generation_with_GPT_2.ipynb
gpt.txt		 sample_data


In [55]:
!jupyter nbconvert --ClearMetadataPreprocessor.enabled=True \
    --to notebook --inplace "Text_Generation_with_GPT_2.ipynb"

[NbConvertApp] Converting notebook Text_Generation_with_GPT_2.ipynb to notebook
[NbConvertApp] Writing 35799 bytes to Text_Generation_with_GPT_2.ipynb
